In [1]:
import sys
sys.path.append("../src")

import fred_api
import analysis
import charts
import db
import summaries

In [2]:
metadata=db.run_query("SELECT max(observation_end) as todays_date FROM raw_fred_metadata")
todays_date=metadata['todays_date'][0]
prompt=summaries.build_timeframe_prompt("Show how prices changed over time in the last 30 years",todays_date)
prompt


'\nTask:\nYour task is to determine the start and end timeframe for the given statement.\n\nAssumptions:\n- Assume today\'s date is 2026-03-01.\n\nDate Interpretation Rules:\n- For year-only references:\n  - Use January 1st as the start date.\n  - Use December 31st as the end date where applicable.\n\n- For relative year-based durations such as:\n  - last 5 years\n  - last 10 years\n  - last 30 years\n\n  infer the start year relative to today\'s date and use January 1st of that inferred year as the start date.\n\n- If no end date is explicitly stated, use today\'s date as the end date.\n\n- If no timeframe can be inferred, default to the most recent 10 calendar years ending on today\'s date.\n\nExamples:\n- "since 2020" → 2020-01-01,2026-03-01\n- "between 2010 and 2020" → 2010-01-01,2020-12-31\n- "in 2024" → 2024-01-01,2024-12-31\n- "last 10 years" → 2016-01-01,2026-03-01\n\nOutput Rules:\n- Return ONLY two values in the format:\n  YYYY-MM-DD,YYYY-MM-DD\n\n- Do not include labels, exp

In [6]:
date_range = summaries.run_prompt(prompt)
start_date,end_date=date_range.split(",")
start_date,end_date

('1996-01-01', '2026-03-01')

In [26]:
# fred_api.build_fred_dataset()
df=db.run_query("SELECT * FROM clean_fred_data")
df.head()

,date,value,series_id
0,1947-01-01,21.48,CPIAUCSL
1,1947-02-01,21.62,CPIAUCSL
2,1947-03-01,22.00,CPIAUCSL
3,1947-04-01,22.00,CPIAUCSL
4,1947-05-01,21.95,CPIAUCSL


In [36]:
metadata=db.run_query("SELECT max(observation_end) as todays_date FROM raw_fred_metadata")
metadata['todays_date'][0]

'2026-03-01'

In [ ]:
PAYEMS_metadata=db.get_series_metadata(series_id="PAYEMS")
PAYEMS_metadata.head()

In [4]:
UNRATE_metadata=db.get_series_metadata(series_id="UNRATE")
UNRATE_metadata.head()

,series_id,title,frequency,units,seasonal_adjustment,observation_start,observation_end,notes
0,UNRATE,Unemployment Rate,Monthly,Percent,Seasonally Adjusted,1948-01-01,2026-03-01,The unemployment rate represents the number of...


In [5]:
unrate_ts = analysis.get_time_series(df,"UNRATE")
unrate_ts.head()

,date,value,series_id,day,month,quarter,year
1887,2026-03-01,4.3,UNRATE,1,3,1,2026
1886,2026-02-01,4.4,UNRATE,1,2,1,2026
1885,2026-01-01,4.3,UNRATE,1,1,1,2026
1884,2025-12-01,4.4,UNRATE,1,12,4,2025
1883,2025-11-01,4.5,UNRATE,1,11,4,2025


In [6]:
# Question asks about levels/averages → pass compute_timeframe_agg()
unrate_yearly_agg=analysis.compute_timeframe_agg(unrate_ts,date_parts=["year"])
unrate_yearly_agg.head()

,year,Average_value,Min_value,Max_value,Median_value,std_value,var_value
78,2026,4.333333,4.3,4.4,4.3,0.057735,0.003333
77,2025,4.263636,4.0,4.5,4.3,0.143337,0.020545
76,2024,4.025000,3.7,4.2,4.1,0.160255,0.025682
75,2023,3.625000,3.4,3.9,3.6,0.142223,0.020227
74,2022,3.650000,3.5,4.0,3.6,0.156670,0.024545


In [7]:
# Question asks about change rate/volatility → pass calculate_percent_change()
unrate_yearly_agg['series_id']='UNRATE'
unrate_yearly_pct_change = analysis.calculate_percent_change(unrate_yearly_agg,index_field="series_id",date_field="year",value_field="Average_value")
unrate_yearly_pct_change.head()

series_id,year,UNRATE
78,2026,1.634684
77,2025,5.928854
76,2024,11.034483
75,2023,-0.684932
74,2022,-31.775701


In [8]:
# Question asks about comparison → pass aggregated comparison table
unrate_payems_wide_df=analysis.compare_series(df[df['series_id'].isin(["PAYEMS","UNRATE"])])
unrate_payems_wide_df.head()

series_id,date,PAYEMS,UNRATE
0,1939-01-01,29923.0,NaN
1,1939-02-01,30100.0,NaN
2,1939-03-01,30280.0,NaN
3,1939-04-01,30094.0,NaN
4,1939-05-01,30299.0,NaN


In [9]:
series_ids=["UNRATE"]

context=""

for series in series_ids:
    metadata=db.get_series_metadata(series_id=series)

    context+=f"""{metadata['title'][0]} dataset. 

    About the dataset:
    {metadata['notes'][0]}
    """

In [10]:
# Question asks about change rate/volatility → pass calculate_percent_change()

#datasets to be summarized
results_preview2015 = summaries.build_stats_preview(unrate_yearly_agg,11)
computed_statistics_preview2015 = summaries.build_stats_preview(unrate_yearly_pct_change,11)

#claude analytical summary
prompt1=summaries.build_prompt("How did unemployment change after 2015?", context, results_preview2015, computed_statistics_preview2015)
summary = summaries.summarize_prompt(prompt1)

print(summary)

# Unemployment Rate Analysis: Post-2015 Trends

## Analytical Summary
Following 2015, the unemployment rate demonstrated a general declining trend from 2016 through 2019, with average annual rates decreasing from 4.88% to 3.68%. This downward trajectory was sharply interrupted in 2020, when the average unemployment rate surged to 8.10% with extreme volatility (standard deviation of 3.62), followed by a sustained recovery through 2023-2024 that brought rates below pre-2020 levels.

## Evidence-Based Insights

• **2016-2019 Contraction Phase**: The unemployment rate declined consistently over a four-year period, with average values decreasing from 4.88% (2016) to 3.68% (2019), representing a 21.3% reduction in the average unemployment rate across this interval.

• **2020 Volatility and Magnitude**: Year 2020 exhibited exceptional disruption with an average unemployment rate of 8.10% and a maximum recorded value of 14.8%, indicating a dramatic spike with substantial intra-year fluctuation

In [11]:
# yearly aggregation narration
results_preview2020 = summaries.build_stats_preview(unrate_yearly_agg,6)
computed_statistics_preview2020 = summaries.build_stats_preview(unrate_yearly_pct_change,6)

#claude analytical summary
prompt2=summaries.build_prompt("how has the average unemployment rate changed since 2020?", context, results_preview2020, computed_statistics_preview2020)
summary = summaries.summarize_prompt(prompt2)

print(summary)

# Unemployment Rate Analysis Summary: Changes Since 2020

## 1. Analytical Summary

The average unemployment rate has declined substantially from 2021 to 2024, decreasing from 5.35% to 4.03%, representing a 1.32 percentage point reduction. This downward trajectory has continued into 2025 and 2026, with average rates of 4.26% and 4.33% respectively, indicating stabilization at levels approximately 1 percentage point below the 2021 peak.

## 2. Evidence-Based Insights

- **Peak volatility occurred in 2021**: The standard deviation of 0.87 in 2021 substantially exceeded subsequent years (0.16 or less), indicating significantly greater monthly fluctuations during this period compared to the relative stability observed from 2022 onward.

- **Consistent year-over-year improvement from 2021-2024**: Average unemployment rates decreased in each successive year during this period (5.35% → 3.65% → 3.63% → 4.03%), with the most substantial decline occurring between 2021 and 2022 (1.7 percentage po

In [12]:
# comparison narration
unrate_monthly_agg=analysis.compute_timeframe_agg(unrate_ts,date_parts=["date"])
results_preview2024 = summaries.build_results_preview(unrate_monthly_agg,"date","2024-01-01",">=")

#claude analytical summary
prompt3=summaries.build_prompt("how has the average unemployment rate changed since 2020?", context, results_preview2024, results_preview2024)
summary = summaries.summarize_prompt(prompt3)

print(summary)


# Analytical Summary

The provided dataset lacks observations from 2020 through 2023, preventing a comprehensive assessment of unemployment rate changes since 2020. The available data spans from January 2024 onwards, showing average unemployment rates ranging from 3.7% to 4.5% during this period, with a general clustering between 4.0% and 4.2% in recent months.

# Evidence-Based Insights

• **Recent stability with minor fluctuation**: From January 2024 through March 2026, the average unemployment rate has remained relatively constrained, oscillating within a 0.8 percentage point range (3.7% to 4.5%), indicating minimal volatility over this 27-month observation window.

• **Slight upward trend in 2025**: Monthly average values in 2025 demonstrate a modest elevation compared to early 2024 baseline observations, with 2025 rates predominantly clustering between 4.1% and 4.5% versus the 3.7%-3.9% range observed in early 2024.

• **Current stabilization in early 2026**: The most recent obser

In [13]:
# ranking narration
unrateyearly2010=analysis.compute_timeframe_agg(unrate_ts,date_parts=["year"])
unrateyearly2010 = summaries.build_results_preview(unrateyearly2010,"year",2010,">=")

unrateyearly2010=analysis.rank_periods(unrateyearly2010, sort_by=["Average_value"], n=16, ascending=True)

#claude analytical summary
prompt4=summaries.build_prompt("rank the lowest unemployment years since 2010", context, unrateyearly2010, unrateyearly2010)
summary = summaries.summarize_prompt(prompt4)

print(summary)

# Unemployment Rate Analysis: Lowest Years Since 2010

## Analytical Summary
Based on average annual unemployment rates, the period from 2018 onwards exhibits the lowest unemployment levels in the dataset, with 2023 recording the minimum average rate at 3.625%. The years 2011-2017 demonstrate substantially elevated unemployment rates, ranging from 4.358% to 8.933%, indicating a marked improvement in labor market conditions over the subsequent decade.

## Evidence-Based Insights

• **2023 ranks as the lowest unemployment year** with an average rate of 3.625% (range: 3.4-3.9%), followed by 2022 at 3.650% and 2019 at 3.675%, demonstrating consistent sub-3.7% averages across this three-year period.

• **The 2018-2019 period marks a structural inflection point**, with 2018 transitioning to lower unemployment (3.892%) compared to the preceding five-year average of 4.86-5.28%, suggesting sustained improvement in labor market tightness post-2017.

• **Variance patterns indicate greater labor m

In [ ]:
# patterns:
# Question asks about levels/averages → pass compute_timeframe_agg()
# Question asks about change rate/volatility → pass calculate_percent_change()
# Question asks about comparison → pass aggregated comparison table

In [ ]:
# context=


print(metadata['title'][0])
print(metadata['notes'][0])
# metadata.head()

In [12]:
# series_ids=["UNRATE"]

# context=""

# for series in series_ids:
#     metadata=db.get_series_metadata(series_id=series)

#     context+=f"""{metadata['title'][0]} dataset. 

#     About the dataset:
#     {metadata['notes'][0]}
#     """

# context_prmopt=f"""You are an analytics narration assistant.  You are summarizing metadata from a dataset to inform other analysis about the context of a given dataset. 

# In 1-2 sentences summarize the {metadata['title'][0]} dataset. 

# About the dataset:
# {metadata['notes'][0]}
# """

# summaries.summarize_prompt(context_prmopt)
# # build_prompt(question, context, results_preview, computed_statistics_preview)

'# Unemployment Rate Dataset Summary\n\nThe Unemployment Rate dataset measures the percentage of unemployed individuals within the civilian labor force aged 16 and older (excluding institutionalized persons and active military), sourced from the Current Population Survey. This U-3 measure of labor underutilization provides a standardized indicator of employment conditions across the 50 states and Washington, D.C.'

In [ ]:
context

In [ ]:
prompt = f"""
    You are an analytics narration assistant. 
    
    You are summarizing the analysis done on the {context}

    Act as a data analysts and use a technical and statistical tone for an audience of other data analysts.
    
    Use ONLY the information provided to inform your answers. Do not invent statistics, trends, causes, or context.
    If you find the data too incomplete or insufficient to make evidence-based insight then say so clearly.
    
    You are not to make any recommendations or actions. 

    Answer the question:
    {question}

    Using the compiled information below.

    Truncated Results:
    {results_preview.to_string(indexed=False)}

    Statistics:
    {computed_statistics_preview.to_string(indexed=False)}

    Return the following items:
    1. A concise 2-3 sentence analytical summary of the provided information. This summary is not non-speculative and grouned in truth.
    2. Three evidence-based insights in bullet points. Keep them statistical and action-oriented.
    3. One caution or limitation of the provided analysis.
    """